In [ ]:
import numpy as np
import scanpy as sc
import multigrate as mtg
import muon

In [ ]:
adata = sc.read_h5ad( "./Data/TEA-seq/TEA-seq.h5ad")
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 25517 × 165454
    obs: 'cell_type', 'batch'
    var: 'modality'
    obsm: 'protein_expression'

In [3]:
adata_gex = adata[:, adata.var['modality'] == "Gene Expression"].copy()
adata_gex.layers['counts'] = adata_gex.X
sc.pp.normalize_total(adata_gex, target_sum=1e4)
sc.pp.log1p(adata_gex)
sc.pp.highly_variable_genes(adata_gex, n_top_genes=4000, batch_key='batch', subset = True)

In [4]:
adata_atac = adata[:, adata.var['modality'] == "Peaks"].copy()
sc.pp.log1p(adata_atac)

In [5]:
adata_adt = sc.AnnData(X = adata.obsm["protein_expression"].astype(float))
adata_adt.obs = adata.obs
muon.prot.pp.clr(adata_adt)
adata_adt.layers['clr']=np.matrix(adata_adt.X.copy())

In [ ]:
adata = mtg.data.organize_multiome_anndatas(
    adatas = [[adata_gex], [adata_atac], [adata_adt]],            # a list of anndata objects per modality, RNA-seq always goes first
    layers = [['counts'], [None], ['clr']],     # if need to use data from .layers, if None use .X
)
mtg.model.MultiVAE.setup_anndata(
    adata,
    categorical_covariate_keys=["batch"],
    rna_indices_end = 4000,
)
model = mtg.model.MultiVAE(
    adata,
    losses=['nb', 'mse', 'mse'],
)
model.train()

model.plot_losses()
model.get_latent_representation()

In [ ]:
np.save('Multigrate_tea_3.npy', adata.obsm['latent'])